In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
crm_customer_df = spark.table("bronze.crm_customers")
erp_customer_df = spark.table("bronze.erp_customers")
erp_location_df = spark.table("bronze.erp_locations")

In [0]:
display(crm_customer_df.limit(5))
display(erp_customer_df.limit(5))
display(erp_location_df.limit(5))

cst_id,cst_key,cst_firstname,cst_lastname,cst_marital_status,cst_gndr,cst_create_date
11000,AW00011000,Jon,Yang,M,M,2025-10-06
11001,AW00011001,Eugene,Huang,S,M,2025-10-06
11002,AW00011002,Ruben,Torres,M,M,2025-10-06
11003,AW00011003,Christy,Zhu,S,F,2025-10-06
11004,AW00011004,Elizabeth,Johnson,S,F,2025-10-06


CID,BDATE,GEN
NASAW00011000,1971-10-06,Male
NASAW00011001,1976-05-10,Male
NASAW00011002,1971-02-09,Male
NASAW00011003,1973-08-14,Female
NASAW00011004,1979-08-05,Female


CID,CNTRY
AW-00011000,Australia
AW-00011001,Australia
AW-00011002,Australia
AW-00011003,Australia
AW-00011004,Australia


In [0]:
crm_customer = (

    crm_customer_df
    .withColumnRenamed("cst_id","customer_id")
    .withColumnRenamed("cst_key","customer_key")
    .withColumnRenamed("cst_firstname","first_name")
    .withColumnRenamed("cst_lastname","last_name")
    .withColumnRenamed("cst_marital_status","marital_status")
    .withColumnRenamed("cst_gndr","gender")
    .withColumnRenamed("cst_create_date","create_date")
)

In [0]:
crm_customer = (
    crm_customer
    .withColumn("first_name",initcap(trim(col("first_name"))))
    .withColumn("last_name",initcap(trim(col("last_name"))))
)

In [0]:
crm_customer = (
    crm_customer
    .withColumn(
        "gender",
        when(upper(col("gender"))=="M","Male")
        .when(upper(col("gender"))=="F","Female")
        .otherwise("Unknown")
    )
)

In [0]:
crm_customer = (
    crm_customer
    .withColumn(
        "marital_status",
        when(upper(col("marital_status"))=="M","Married")
        .when(upper(col("marital_status"))=="S","Single")
        .otherwise("Unknown")
    )
)

In [0]:
crm_customer = (
    crm_customer
    .withColumn(
        "create_date",
        to_date(col("create_date"))
    )
)

In [0]:

erp_customer = (
    erp_customer_df
    .withColumnRenamed("CID","customer_key")
    .withColumnRenamed("BDATE","birth_date")
    .withColumnRenamed("GEN","erp_gender")
)

In [0]:
erp_customer = (
    erp_customer
    .withColumn(
        "customer_key",
        regexp_replace(
            col("customer_key"),
            "^NAS",
            ""
        )
    )
)

In [0]:
erp_customer = (
    erp_customer
    .withColumn(
        "erp_gender",
        initcap(trim(col("erp_gender")))
    )
)

In [0]:
erp_customer = (
    erp_customer
    .withColumn(
        "birth_date",
        to_date(col("birth_date"))
    )
)

In [0]:
erp_location = (
    erp_location_df
    .withColumnRenamed("CID","customer_key")
    .withColumnRenamed("CNTRY","country")
)

In [0]:
erp_location = (
    erp_location
    .withColumn(
        "customer_key",
        regexp_replace(
            col("customer_key"),
            "-",
            ""
        )
    )
)

In [0]:
customer_df = (
    crm_customer.alias("crm")
    .join(
        erp_customer.alias("erp"),
        "customer_key",
        "left"
    )
)

In [0]:
customer_df = (
    customer_df.alias("cust")
    .join(
        erp_location.alias("loc"),
        "customer_key",
        "left"
    )
)

In [0]:
customer_df = (
    customer_df
    .withColumn(
        "gender",
        coalesce(
            col("gender"),
            col("erp_gender")
        )
    )
)

In [0]:
customer_df = (
    customer_df
    .dropDuplicates(
        ["customer_id"]
    )
)

In [0]:
customer_df = customer_df.select(
    "customer_id",
    "customer_key",
    "first_name",
    "last_name",
    "birth_date",
    "gender",
    "marital_status",
    "country",
    "create_date"
)

In [0]:
print("CUSTOMER TABLE SUMMARY")
print("="*70)
print("Rows :",customer_df.count())
print("Columns :",len(customer_df.columns))
display(customer_df.limit(10))

CUSTOMER TABLE SUMMARY
Rows : 18485
Columns : 9


customer_id,customer_key,first_name,last_name,birth_date,gender,marital_status,country,create_date
11016,AW00011016,Wyatt,Hill,1984-10-25,Unknown,Married,US,2025-10-07
11033,AW00011033,Jaime,Nath,1958-09-19,Unknown,Married,Australia,2025-10-07
11044,AW00011044,Adam,Flores,1954-11-21,Unknown,Married,Australia,2025-10-07
11052,AW00011052,Heidi,Lopez,1957-02-03,Unknown,Single,Australia,2025-10-07
11082,AW00011082,Angela,Butler,1972-02-01,Unknown,Single,US,2025-10-07
11086,AW00011086,Ryan,Brown,1963-06-22,Unknown,Married,US,2025-10-07
11108,AW00011108,Kari,Alvarez,1969-01-10,Unknown,Single,Australia,2025-10-07
11125,AW00011125,Dana,Navarro,1961-10-06,Unknown,Single,Australia,2025-10-07
11153,AW00011153,Angela,James,1981-12-21,Unknown,Married,US,2025-10-07
11169,AW00011169,Bryce,Richardson,1973-12-20,Unknown,Married,US,2025-10-07


In [0]:
# COMMAND ----------

spark.sql("CREATE DATABASE IF NOT EXISTS silver")

DataFrame[]

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

DataFrame[]

In [0]:
(
customer_df.write
.format("delta")
.mode("overwrite")
.saveAsTable(
"silver.customers"
)
)

In [0]:
spark.sql("SHOW TABLES IN silver").show()

display(spark.table("silver.customers"))

+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
|  silver|customers|      false|
+--------+---------+-----------+



customer_id,customer_key,first_name,last_name,birth_date,gender,marital_status,country,create_date
11016,AW00011016,Wyatt,Hill,1984-10-25,Unknown,Married,US,2025-10-07
11033,AW00011033,Jaime,Nath,1958-09-19,Unknown,Married,Australia,2025-10-07
11044,AW00011044,Adam,Flores,1954-11-21,Unknown,Married,Australia,2025-10-07
11052,AW00011052,Heidi,Lopez,1957-02-03,Unknown,Single,Australia,2025-10-07
11082,AW00011082,Angela,Butler,1972-02-01,Unknown,Single,US,2025-10-07
11086,AW00011086,Ryan,Brown,1963-06-22,Unknown,Married,US,2025-10-07
11108,AW00011108,Kari,Alvarez,1969-01-10,Unknown,Single,Australia,2025-10-07
11125,AW00011125,Dana,Navarro,1961-10-06,Unknown,Single,Australia,2025-10-07
11153,AW00011153,Angela,James,1981-12-21,Unknown,Married,US,2025-10-07
11169,AW00011169,Bryce,Richardson,1973-12-20,Unknown,Married,US,2025-10-07
